# Analysis Notebook: Evaluating Feature Impact on Soil Classification

This notebook explores the impact of various image processing features on soil classification. We will visualize feature distributions, examine correlations, compare features across soil classes, and perform a feature importance analysis using a Random Forest model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder

%matplotlib inline

# Load your dataset (update the filename/path as necessary)
df = pd.read_csv('soil_data.csv')

# Display the first few rows of the dataset
df.head()

## 1. Distribution of Each Feature

Visualize each feature’s distribution to understand its range and skewness.

In [ ]:
# List of feature columns (adjust these to match your dataset)
feature_cols = [
    'canny_edge_strength', 
    'block_sum_10x10_percentile', 
    'block_sum_20x20_percentile', 
    'fourier_percentile', 
    'fourier_radius', 
    'num_connected_components', 
    'blending_ratio', 
    'cannyx', 
    'cannyy'
]

# Plot histograms for each feature
df[feature_cols].hist(bins=20, figsize=(15, 10))
plt.tight_layout()
plt.suptitle("Distribution of Each Feature", y=1.02)
plt.show()

## 2. Correlation Heatmap of Features

Generate a correlation heatmap to identify any strong linear relationships among features.

In [ ]:
# Compute the correlation matrix
corr = df[feature_cols].corr()

# Plot the correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap of Features")
plt.show()

## 3. Feature Distribution by Soil Class

Compare feature distributions across different soil classes using box plots. (Assuming the target variable is named `soil_class`.)

In [ ]:
# Assuming 'soil_class' is the target variable
plt.figure(figsize=(15, 10))
for i, col in enumerate(feature_cols):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(x='soil_class', y=col, data=df)
    plt.title(f'{col} by Soil Class')
    plt.xlabel('Soil Class')
    plt.ylabel(col)
plt.tight_layout()
plt.show()

## 4. Effect of Histogram Equalization Blending Ratio on Model Performance

Explore how varying blending ratios relate to model performance metrics.

In [ ]:
# If you have model metric scores saved in the dataset (e.g., 'model_metric')
if 'model_metric' in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='blending_ratio', y='model_metric', hue='soil_class', data=df)
    plt.title('Blending Ratio vs. Model Metric')
    plt.xlabel('Blending Ratio')
    plt.ylabel('Model Metric Score')
    plt.show()
else:
    print("Column 'model_metric' not found in dataset. Please add model performance metrics to analyze this relationship.")

## 5. Distribution of Canny Edge Parameters

Visualize the distributions of the canny edge thresholds (`cannyx` and `cannyy`).

In [ ]:
plt.figure(figsize=(12, 5))

# Distribution of cannyx values
plt.subplot(1, 2, 1)
sns.histplot(df['cannyx'], bins=20, kde=True)
plt.title('Distribution of CannyX')

# Distribution of cannyy values
plt.subplot(1, 2, 2)
sns.histplot(df['cannyy'], bins=20, kde=True)
plt.title('Distribution of CannyY')

plt.tight_layout()
plt.show()

## 6. Comparison of Naive Summation Block Sizes (10x10 vs. 20x20)

Compare the distributions of block-sum percentiles for 10x10 and 20x20 blocks.

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(df['block_sum_10x10_percentile'], label='10x10 Block', shade=True)
sns.kdeplot(df['block_sum_20x20_percentile'], label='20x20 Block', shade=True)
plt.title('Density Plot: Block Sum Percentiles for 10x10 vs. 20x20 Blocks')
plt.xlabel('Percentile Value')
plt.ylabel('Density')
plt.legend()
plt.show()

## 7. Analysis of Fourier Transform Features with Respect to Radius

Investigate the relationship between Fourier-based percentiles and the selected central radius.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='fourier_radius', y='fourier_percentile', hue='soil_class', data=df)
plt.title('Fourier Percentile vs. Fourier Radius')
plt.xlabel('Fourier Radius')
plt.ylabel('Fourier Percentile')
plt.show()

## 8. Relationship of Number of Connected Components Across Soil Classes

Assess whether connected component counts differ significantly across soil types.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='soil_class', y='num_connected_components', data=df)
plt.title('Number of Connected Components by Soil Class')
plt.xlabel('Soil Class')
plt.ylabel('Number of Connected Components')
plt.show()

## 9. Pairwise Relationships Among Features

Use pairplots to visually inspect relationships among features and observe potential clustering.

In [ ]:
sns.pairplot(df[feature_cols + ['soil_class']], hue='soil_class', diag_kind='kde')
plt.suptitle('Pairwise Relationships Among Features', y=1.02)
plt.show()

## 10. Feature Importance Analysis Using Random Forest and Permutation Importance

Apply a Random Forest model to rank features by importance and use permutation importance for further validation.

In [ ]:
# Prepare the data for modeling
X = df[feature_cols]
y = df['soil_class']

# Encode the target variable if it is categorical
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Fit a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y_encoded)

# Get feature importances from the Random Forest
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=np.array(feature_cols)[indices], y=importances[indices])
plt.title('Feature Importances from Random Forest')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=45)
plt.show()

# Permutation Importance Analysis
result = permutation_importance(rf, X, y_encoded, n_repeats=10, random_state=42)
sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(10, 6))
sns.barplot(x=np.array(feature_cols)[sorted_idx], y=result.importances_mean[sorted_idx])
plt.title('Permutation Feature Importances')
plt.xlabel('Feature')
plt.ylabel('Mean Importance')
plt.xticks(rotation=45)
plt.show()

This concludes our analysis. Adjust the code as needed based on your dataset and specific project requirements.